# 🗺️ Vector Data Processing Using Python — Exercise Notebook

---

> **How to use this notebook:**  
> Each section contains **worked examples** you can run, followed by **practice exercises**.  
> Read the examples carefully, then attempt the exercises in the solution cells.  
> Use `Shift + Enter` to run a cell.

---

## 📋 Table of Contents

| # | Topic |
|---|-------|
| 0 | Setup & Environment |
| 1 | Reading Vector Data (Shapefiles & GeoJSON) |
| 2 | Exploring & Inspecting a GeoDataFrame |
| 3 | Filtering & Subsetting Spatial Data |
| 4 | Reprojecting for Accurate Analysis |
| 5 | Spatial Operations & Attribute Manipulation |
| 6 | Visualizing with Matplotlib |
| 7 | Multi-Panel County Maps (Automation) |
| 8 | Exporting Results |
| 9 | Census Demographic Data with pygris |
| 10 | Choropleth Mapping with Census Data |
| 11 | Bonus — Spatial Joins & Dissolve |

---

# 0️⃣ Setup & Environment

---

> This notebook uses **Google Colab** as the primary environment.  
> Data: North Dakota TIGER census tract shapefiles and U.S. Census ACS data.

| Library | Purpose |
|---------|----------|
| `geopandas` | Read, process, and write vector geospatial data |
| `shapely` | Geometry operations (buffer, centroid, intersect) |
| `matplotlib` | Static map visualization |
| `numpy` | Numerical operations on attribute data |
| `pygris` | Access U.S. Census TIGER and ACS data |
| `fiona` | Underlying file I/O (used by GeoPandas) |

---

### 💡 Example 0.1 — Mount Drive & Create Folders

In [ ]:
# Mount Google Drive (run once per session)
from google.colab import drive
drive.mount('/content/drive')

import os
data_folder   = 'data'
output_folder = 'output'

for folder in [data_folder, output_folder]:
    if not os.path.exists(folder):
        os.mkdir(folder)
        print(f'Created: {folder}')
    else:
        print(f'Already exists: {folder}')

### 💡 Example 0.2 — Import All Required Libraries

In [ ]:
# !pip install geopandas shapely matplotlib numpy pygris   # uncomment if needed

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np

print('GeoPandas version :', gpd.__version__)
print('All libraries loaded ✅')
!pwd

### ✏️ Exercise 0 — Practice Questions

**Q1.** Print the installed versions of `geopandas`, `shapely`, and `matplotlib`.

**Q2.** Create an additional sub-folder `output/maps` using `os.makedirs(..., exist_ok=True)`.

**Q3.** Run `!pip list | grep -E 'geopandas|shapely|fiona|pyproj'` to confirm all GeoPandas dependencies are installed.

**Q4.** List all files currently in your `data` folder using `os.listdir()`. What formats are present?

**Q5.** What does `!pwd` print? What working directory does Colab use by default?


In [ ]:
# 📝 Write your solutions here

# Q1:

# Q2:

# Q3:

# Q4:

# Q5:


<details>
<summary>🔍 Click to reveal solutions</summary>

```python
import geopandas as gpd, shapely, matplotlib, os

# Q1:
import shapely
print('GeoPandas :', gpd.__version__)
print('Shapely   :', shapely.__version__)
import matplotlib; print('Matplotlib:', matplotlib.__version__)

# Q2:
os.makedirs('output/maps', exist_ok=True)
print('output/maps ready')

# Q3:
# !pip list | grep -E 'geopandas|shapely|fiona|pyproj'

# Q4:
print(os.listdir('data'))
# Typical formats: .zip (shapefile), .shp, .geojson, .tif

# Q5:
# !pwd  → /content  (default Colab working directory)
```
</details>

---

# 1️⃣ Reading Vector Data

---

> **GeoPandas** reads virtually any vector format into a `GeoDataFrame` — a table with a special `geometry` column.  
> `gpd.read_file()` auto-detects format using **Fiona / GDAL** under the hood.

| Format | Extension | Notes |
|--------|-----------|-------|
| ESRI Shapefile | `.shp` / `.zip` | Most common; requires multiple sidecar files |
| GeoJSON | `.geojson` | Human-readable; single file |
| GeoPackage | `.gpkg` | Modern, single-file alternative to Shapefile |
| Parquet | `.parquet` | Fast, columnar; great for large datasets |
| KML | `.kml` | Google Earth format |

---

### 💡 Example 1.1 — Read a Zipped Shapefile

In [ ]:
import geopandas as gpd

ShapefilePath = r'data/tl_2025_38_tract.zip'

# gpd.read_file() auto-detects the format and reads via Fiona
StateNd = gpd.read_file(ShapefilePath)

# Display the first 5 rows — note the special 'geometry' column
print(f'Type  : {type(StateNd)}')
print(f'Shape : {StateNd.shape}  (rows × columns)')
print(f'Columns: {StateNd.columns.tolist()}')
StateNd.head(5)

### 💡 Example 1.2 — Read GeoJSON and GeoPackage

In [ ]:
import geopandas as gpd

# Reading a GeoJSON (same function, different format)
# gdf_json = gpd.read_file('data/my_layer.geojson')

# Reading a GeoPackage layer
# gdf_gpkg = gpd.read_file('data/my_data.gpkg', layer='layer_name')

# Reading directly from a URL (e.g. GitHub raw GeoJSON)
url = 'https://raw.githubusercontent.com/datasets/geo-countries/master/data/countries.geojson'
try:
    world = gpd.read_file(url)
    print(f'World countries loaded: {len(world)} features')
    world.head(3)
except Exception as e:
    print('URL read skipped (no internet):', e)

### 💡 Example 1.3 — Create a GeoDataFrame from Scratch

In [ ]:
import geopandas as gpd
from shapely.geometry import Point, Polygon
import pandas as pd

# From a list of Shapely geometries
cities = gpd.GeoDataFrame({
    'city'      : ['Fargo', 'Bismarck', 'Grand Forks', 'Minot'],
    'population': [125_000, 74_000, 59_000, 48_000],
    'geometry'  : [
        Point(-96.79, 46.88),
        Point(-100.78, 46.81),
        Point(-97.03, 47.93),
        Point(-101.30, 48.23)
    ]
}, crs='EPSG:4326')

print(cities)
cities.plot(markersize=80, color='red', figsize=(6, 4))
plt.title('Major ND Cities')
plt.show()

### ✏️ Exercise 1 — Practice Questions

**Q1.** Read `tl_2025_38_tract.zip` into a GeoDataFrame. Print its `shape`, `crs`, and list of column names.

**Q2.** Create a GeoDataFrame of **5 landmarks in your country** with columns `name`, `type`, and `geometry` (Points). Set the CRS to EPSG:4326.

**Q3.** Save the GeoDataFrame from Q2 to `output/landmarks.geojson` using `to_file()`. Re-read it and confirm the row count matches.

**Q4.** Read the same shapefile **twice** into two variables. Confirm they have the same `.shape` and `.crs`.

**Q5.** What exception is raised if you call `gpd.read_file()` on a non-existent path? Try it with a `try/except` block and print the error message.


In [ ]:
# 📝 Write your solutions here

# Q1:

# Q2:

# Q3:

# Q4:

# Q5:


<details>
<summary>🔍 Click to reveal solutions</summary>

```python
import geopandas as gpd
from shapely.geometry import Point

# Q1:
gdf = gpd.read_file('data/tl_2025_38_tract.zip')
print(gdf.shape, gdf.crs, gdf.columns.tolist())

# Q2:
landmarks = gpd.GeoDataFrame({
    'name': ['Pashupatinath','Boudhanath','Swayambhunath','Patan Durbar','Bhaktapur'],
    'type': ['Temple','Stupa','Stupa','Palace','Palace'],
    'geometry': [Point(85.349,27.710),Point(85.362,27.721),
                 Point(85.290,27.715),Point(85.325,27.672),Point(85.428,27.671)]
}, crs='EPSG:4326')
print(landmarks)

# Q3:
landmarks.to_file('output/landmarks.geojson', driver='GeoJSON')
lm2 = gpd.read_file('output/landmarks.geojson')
print('Match:', len(landmarks)==len(lm2))

# Q4:
gdf1 = gpd.read_file('data/tl_2025_38_tract.zip')
gdf2 = gpd.read_file('data/tl_2025_38_tract.zip')
print(gdf1.shape==gdf2.shape, gdf1.crs==gdf2.crs)

# Q5:
try:
    gpd.read_file('data/nonexistent.shp')
except Exception as e:
    print(type(e).__name__, ':', e)
```
</details>

---

# 2️⃣ Exploring & Inspecting a GeoDataFrame

---

> A `GeoDataFrame` inherits all Pandas methods **plus** spatial properties.  
> Use these methods to understand the data before any analysis.

| Method / Property | Description |
|-------------------|-------------|
| `.head(n)` / `.tail(n)` | First / last n rows |
| `.info()` | Column names, dtypes, non-null counts |
| `.describe()` | Descriptive statistics for numeric columns |
| `.crs` | Coordinate Reference System |
| `.geom_type` | Geometry type per feature |
| `.total_bounds` | Spatial extent `[minx, miny, maxx, maxy]` |
| `['col'].nunique()` | Number of unique values in a column |
| `['col'].value_counts()` | Frequency of each value |

---

### 💡 Example 2.1 — Basic Metadata Inspection

In [ ]:
import geopandas as gpd

StateNd = gpd.read_file(r'data/tl_2025_38_tract.zip')

print('=== Shape ===')
print(f'  Rows    : {StateNd.shape[0]}')
print(f'  Columns : {StateNd.shape[1]}')

print('\n=== CRS ===')
print(f'  {StateNd.crs}')

print('\n=== Geometry Types ===')
print(f'  {set(StateNd.geom_type)}')

print('\n=== Spatial Extent ===')
print(f'  {StateNd.total_bounds}')

print('\n=== Column Data Types ===')
print(StateNd.dtypes)

### 💡 Example 2.2 — Attribute Statistics

In [ ]:
import geopandas as gpd

StateNd = gpd.read_file(r'data/tl_2025_38_tract.zip')

# Number of unique counties
n_counties = StateNd['COUNTYFP'].nunique()
print(f'Unique counties   : {n_counties}')

# Number of census tracts
print(f'Total tracts      : {len(StateNd)}')

# Top 5 counties by number of tracts
print('\nTop 5 counties by tract count:')
print(StateNd['COUNTYFP'].value_counts().head())

# Descriptive stats on ALAND (land area in m²)
print('\nALAND (land area m²) stats:')
print(StateNd['ALAND'].describe())

### 💡 Example 2.3 — Detailed DataFrame Info

In [ ]:
import geopandas as gpd

StateNd = gpd.read_file(r'data/tl_2025_38_tract.zip')

# Full column info (dtype + non-null count)
StateNd.info()

# Check for any missing values
print('\nMissing values per column:')
print(StateNd.isnull().sum())

### ✏️ Exercise 2 — Practice Questions

**Q1.** Print the **total number of tracts**, **unique county codes**, and **unique geometry types** in the ND shapefile.

**Q2.** Which county has the **most census tracts**? Which has the **fewest**? Print both county codes and counts.

**Q3.** Print the **spatial bounding box** of the dataset (`total_bounds`). What are the approximate lat/lon limits of North Dakota?

**Q4.** Compute and print summary statistics (min, max, mean, median) for the `ALAND` column (land area in m²). Convert the mean to km².

**Q5.** Check for **missing values** in every column. Which columns, if any, have null entries?


In [ ]:
# 📝 Write your solutions here

# Q1:

# Q2:

# Q3:

# Q4:

# Q5:


<details>
<summary>🔍 Click to reveal solutions</summary>

```python
import geopandas as gpd, numpy as np
StateNd = gpd.read_file('data/tl_2025_38_tract.zip')

# Q1:
print('Tracts:', len(StateNd))
print('Counties:', StateNd['COUNTYFP'].nunique())
print('Geom types:', set(StateNd.geom_type))

# Q2:
vc = StateNd['COUNTYFP'].value_counts()
print('Most tracts :', vc.idxmax(), vc.max())
print('Fewest tracts:', vc.idxmin(), vc.min())

# Q3:
minx,miny,maxx,maxy = StateNd.total_bounds
print(f'Lon: {minx:.2f} to {maxx:.2f}, Lat: {miny:.2f} to {maxy:.2f}')

# Q4:
aland = StateNd['ALAND']
print(f'Min: {aland.min()/1e6:.1f} km²  Max: {aland.max()/1e6:.1f} km²')
print(f'Mean: {aland.mean()/1e6:.1f} km²  Median: {aland.median()/1e6:.1f} km²')

# Q5:
print(StateNd.isnull().sum()[StateNd.isnull().sum()>0])
```
</details>

---

# 3️⃣ Filtering & Subsetting Spatial Data

---

> GeoPandas inherits Pandas **boolean indexing** — filter rows by attribute conditions exactly as you would a DataFrame.

| Operation | Code |
|-----------|------|
| Single value | `gdf[gdf['col'] == 'value']` |
| Multiple values | `gdf[gdf['col'].isin(['a','b'])]` |
| Numeric range | `gdf[(gdf['col'] > 100) & (gdf['col'] < 500)]` |
| String contains | `gdf[gdf['col'].str.contains('text')]` |
| Select columns | `gdf[['col1','col2','geometry']]` |
| Reset index | `.reset_index(drop=True)` |

---

### 💡 Example 3.1 — Filter by County Code

In [ ]:
import geopandas as gpd

StateNd = gpd.read_file(r'data/tl_2025_38_tract.zip')

# Cass County = COUNTYFP '017'
CassCounty = StateNd[StateNd['COUNTYFP'] == '017'].copy()
print(f'Cass County tracts : {len(CassCounty)}')
CassCounty.head(3)

### 💡 Example 3.2 — Filter by Multiple Conditions

In [ ]:
import geopandas as gpd

StateNd = gpd.read_file(r'data/tl_2025_38_tract.zip')

# Select multiple counties
selected_counties = ['017', '035', '057']
multi_county = StateNd[StateNd['COUNTYFP'].isin(selected_counties)].copy()
print(f'Selected counties row count: {len(multi_county)}')

# Filter by land area: tracts larger than 500 km²
large_tracts = StateNd[StateNd['ALAND'] > 500_000_000].copy()
print(f'Tracts > 500 km²: {len(large_tracts)}')

# Combine: large tracts in Cass County
cass_large = StateNd[
    (StateNd['COUNTYFP'] == '017') & (StateNd['ALAND'] > 50_000_000)
].copy()
print(f'Large tracts in Cass County: {len(cass_large)}')

### 💡 Example 3.3 — Select Specific Columns & Reset Index

In [ ]:
import geopandas as gpd

StateNd = gpd.read_file(r'data/tl_2025_38_tract.zip')

# Keep only relevant columns
subset = StateNd[['GEOID', 'COUNTYFP', 'TRACTCE', 'ALAND', 'AWATER', 'geometry']].copy()
print('Subset columns:', subset.columns.tolist())

# Filter and reset index so rows are numbered from 0
cass = subset[subset['COUNTYFP'] == '017'].reset_index(drop=True)
print(f'\nCass County subset, index reset:')
print(cass.head(3))

### ✏️ Exercise 3 — Practice Questions

**Q1.** Filter the ND GeoDataFrame to **Burleigh County** (`COUNTYFP == '015'`). How many tracts does it have?

**Q2.** Find all tracts where the **water area** (`AWATER`) is greater than **1,000,000 m²** (1 km²). How many are there?

**Q3.** Select tracts from **both** Cass County (`017`) **and** Grand Forks County (`035`). Create a plot comparing them side by side using `plt.subplots(1, 2)`.

**Q4.** Filter tracts where `ALAND` is in the **top 10% of land area** across the entire state. Print the count and list the county codes.

**Q5.** Create a subset GeoDataFrame with only the columns `GEOID`, `COUNTYFP`, `ALAND`, and `geometry`. Save it to `output/nd_subset.shp`.


In [ ]:
# 📝 Write your solutions here

# Q1:

# Q2:

# Q3:

# Q4:

# Q5:


<details>
<summary>🔍 Click to reveal solutions</summary>

```python
import geopandas as gpd, numpy as np, matplotlib.pyplot as plt
StateNd = gpd.read_file('data/tl_2025_38_tract.zip')

# Q1:
burleigh = StateNd[StateNd['COUNTYFP']=='015']
print('Burleigh tracts:', len(burleigh))

# Q2:
water = StateNd[StateNd['AWATER']>1_000_000]
print('High-water tracts:', len(water))

# Q3:
cass = StateNd[StateNd['COUNTYFP']=='017']
gf   = StateNd[StateNd['COUNTYFP']=='035']
fig,(ax1,ax2) = plt.subplots(1,2,figsize=(12,5))
cass.plot(ax=ax1,edgecolor='k',facecolor='lightyellow'); ax1.set_title('Cass County')
gf.plot(ax=ax2,edgecolor='k',facecolor='lightblue');   ax2.set_title('Grand Forks County')
plt.tight_layout(); plt.show()

# Q4:
p90 = np.percentile(StateNd['ALAND'],90)
top10 = StateNd[StateNd['ALAND']>=p90]
print('Top 10% count:', len(top10))
print(top10['COUNTYFP'].value_counts())

# Q5:
sub = StateNd[['GEOID','COUNTYFP','ALAND','geometry']].copy()
sub.to_file('output/nd_subset.shp')
print('Saved nd_subset.shp')
```
</details>

---

# 4️⃣ Reprojecting for Accurate Analysis

---

> **CRS matters**: Geographic CRS (degrees) → distances and areas are meaningless.  
> Always reproject to a **projected CRS** (metres) before spatial calculations.

| CRS | EPSG | Units | Best For |
|-----|------|-------|----------|
| WGS 84 | 4326 | Degrees | Storage, web mapping |
| NAD83 | 4269 | Degrees | US Census default |
| US Albers (CONUS) | 5070 | Metres | Area calculations (CONUS) |
| UTM Zone 14N | 32614 | Metres | Local / state accuracy |
| Web Mercator | 3857 | Metres (distorted) | Tile-based web maps |

---

### 💡 Example 4.1 — Reproject to EPSG:5070

In [ ]:
import geopandas as gpd

StateNd = gpd.read_file(r'data/tl_2025_38_tract.zip')

print('Original CRS :', StateNd.crs)

# Reproject to US Albers Equal Area (metres)
StateNdProj = StateNd.to_crs(epsg=5070)

print('Projected CRS:', StateNdProj.crs)
print('\nOriginal bounds  :', StateNd.total_bounds.round(4))
print('Projected bounds :', StateNdProj.total_bounds.round(0))

### 💡 Example 4.2 — Why Projection Matters for Area

In [ ]:
import geopandas as gpd

StateNd = gpd.read_file(r'data/tl_2025_38_tract.zip')

# Area in geographic CRS (degrees² — meaningless)
area_deg = StateNd.geometry.area

# Area after projecting to metres
StateNdProj = StateNd.to_crs(epsg=5070)
area_m2 = StateNdProj.geometry.area

print('Area in degrees² (first 3):' , area_deg.head(3).values)
print('Area in m²       (first 3):' , area_m2.head(3).values.round(0))
print('Area in km²      (first 3):' , (area_m2.head(3)/1e6).values.round(2))

### ✏️ Exercise 4 — Practice Questions

**Q1.** Reproject `StateNd` to **EPSG:32614** (UTM Zone 14N). Print the new CRS and bounding box.

**Q2.** Compare the **centroid coordinates** (x, y) of the first tract in EPSG:4269 vs EPSG:5070. What are the units in each?

**Q3.** Calculate the **total land area of North Dakota** in km² by summing `ALAND` after projecting to EPSG:5070. Compare to the known value (~183,108 km²).

**Q4.** Why should you NOT compute `gdf.geometry.length` on a geographic CRS like EPSG:4326? What would the values represent?

**Q5.** Reproject the GeoDataFrame to **Web Mercator (EPSG:3857)**. Compute the area of one tract and explain why it may be inaccurate for North Dakota.


In [ ]:
# 📝 Write your solutions here

# Q1:

# Q2:

# Q3:

# Q4:

# Q5:


<details>
<summary>🔍 Click to reveal solutions</summary>

```python
import geopandas as gpd
StateNd = gpd.read_file('data/tl_2025_38_tract.zip')

# Q1:
utm = StateNd.to_crs(epsg=32614)
print(utm.crs, utm.total_bounds.round(0))

# Q2:
c_geo  = StateNd.geometry.iloc[0].centroid
c_proj = StateNd.to_crs(5070).geometry.iloc[0].centroid
print(f'Geographic centroid : ({c_geo.x:.4f}°, {c_geo.y:.4f}°)')
print(f'Projected centroid  : ({c_proj.x:.0f} m, {c_proj.y:.0f} m)')

# Q3:
nd_proj = StateNd.to_crs(5070)
total_km2 = nd_proj['ALAND'].sum()/1e6
print(f'Total ALAND: {total_km2:,.0f} km² (known ~183,108 km²)')

# Q4:
# In EPSG:4326, length is in degrees — not meaningful for real-world distances.
# 1 degree ≠ constant metres (varies with latitude).

# Q5:
wm = StateNd.to_crs(3857)
print('First tract area (Web Mercator):', wm.geometry.iloc[0].area/1e6, 'km²')
# Web Mercator distorts area significantly at high latitudes (ND ~47°N) → inaccurate.
```
</details>

---

# 5️⃣ Spatial Operations & Attribute Manipulation

---

> GeoPandas exposes **Shapely geometry methods** on the entire GeoSeries column at once.

| Operation | Code | Returns |
|-----------|------|---------|
| Area | `gdf.geometry.area` | GeoSeries of floats |
| Length / Perimeter | `gdf.geometry.length` | GeoSeries of floats |
| Centroid | `gdf.geometry.centroid` | GeoSeries of Points |
| Bounding box | `gdf.geometry.envelope` | GeoSeries of Polygons |
| Buffer | `gdf.geometry.buffer(dist)` | GeoSeries of Polygons |
| Convex hull | `gdf.geometry.convex_hull` | GeoSeries of Polygons |
| Simplify | `gdf.geometry.simplify(tol)` | GeoSeries |
| Is valid | `gdf.geometry.is_valid` | GeoSeries of bools |

---

### 💡 Example 5.1 — Calculate Area & Centroid

In [ ]:
import geopandas as gpd

StateNd = gpd.read_file(r'data/tl_2025_38_tract.zip')
StateNdProj = StateNd.to_crs(epsg=5070)

# Area in m² and km²
StateNdProj['AreaM2']  = StateNdProj.geometry.area
StateNdProj['AreaKm2'] = StateNdProj['AreaM2'] / 1_000_000

# Centroid (also in projected metres)
StateNdProj['Centroid'] = StateNdProj.geometry.centroid

# Perimeter
StateNdProj['PerimM'] = StateNdProj.geometry.length

print(StateNdProj[['GEOID','COUNTYFP','AreaKm2','PerimM']].head())
print(f'\nLargest tract : {StateNdProj["AreaKm2"].max():.1f} km²')
print(f'Smallest tract: {StateNdProj["AreaKm2"].min():.2f} km²')

### 💡 Example 5.2 — Buffer, Envelope & Convex Hull

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt

StateNd = gpd.read_file(r'data/tl_2025_38_tract.zip')
cass    = StateNd[StateNd['COUNTYFP']=='017'].to_crs(5070)

# Dissolve all Cass County tracts into one polygon
cass_union   = cass.dissolve()

# Buffer: expand boundary by 5 km
cass_buffer  = cass_union.copy()
cass_buffer['geometry'] = cass_union.geometry.buffer(5000)

# Convex hull: smallest convex polygon containing all features
cass_hull = cass_union.copy()
cass_hull['geometry'] = cass_union.geometry.convex_hull

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, gdf, title in zip(axes,
    [cass_union, cass_buffer, cass_hull],
    ['Dissolved Union', '5 km Buffer', 'Convex Hull']):
    gdf.plot(ax=ax, facecolor='lightblue', edgecolor='navy')
    ax.set_title(title); ax.set_axis_off()
plt.tight_layout()
plt.show()

### 💡 Example 5.3 — Validate & Fix Geometries

In [ ]:
import geopandas as gpd

StateNd = gpd.read_file(r'data/tl_2025_38_tract.zip')

# Check for invalid geometries
invalid = StateNd[~StateNd.geometry.is_valid]
print(f'Invalid geometries: {len(invalid)}')

# Fix invalid geometries with buffer(0) trick
StateNd['geometry'] = StateNd.geometry.buffer(0)
print(f'Invalid after fix : {(~StateNd.geometry.is_valid).sum()}')

# Check for empty geometries
empty = StateNd[StateNd.geometry.is_empty]
print(f'Empty geometries  : {len(empty)}')

### ✏️ Exercise 5 — Practice Questions

**Q1.** For the projected ND GeoDataFrame, add a column `'CompactRatio'` defined as `4π × Area / Perimeter²`. A circle has ratio = 1; complex shapes are close to 0. Which tract is most compact?

**Q2.** Buffer all Cass County tracts by **2 km** individually (before dissolving). Plot the original tracts and their buffers overlaid.

**Q3.** Compute the **centroid GeoDataFrame** (as a new GeoDataFrame with the centroids as geometry). Plot the centroids coloured by county.

**Q4.** Use `.simplify(tolerance=1000)` (1 km) on the projected ND layer. Compare file sizes when saved to shapefiles (original vs simplified).

**Q5.** For each county, calculate the **total land area** in km² using `groupby` + `sum`. Print the top 5 counties by total area.


In [ ]:
# 📝 Write your solutions here

# Q1:

# Q2:

# Q3:

# Q4:

# Q5:


<details>
<summary>🔍 Click to reveal solutions</summary>

```python
import geopandas as gpd, numpy as np, matplotlib.pyplot as plt, os
StateNd    = gpd.read_file('data/tl_2025_38_tract.zip')
StateNdProj = StateNd.to_crs(5070)

# Q1:
StateNdProj['AreaM2'] = StateNdProj.geometry.area
StateNdProj['PerimM'] = StateNdProj.geometry.length
StateNdProj['Compact'] = (4*np.pi*StateNdProj['AreaM2']) / StateNdProj['PerimM']**2
print('Most compact:', StateNdProj.loc[StateNdProj['Compact'].idxmax(),'GEOID'])

# Q2:
cass = StateNdProj[StateNdProj['COUNTYFP']=='017'].copy()
buf  = cass.copy(); buf['geometry'] = cass.geometry.buffer(2000)
fig,ax = plt.subplots(figsize=(8,8))
buf.plot(ax=ax,facecolor='lightyellow',edgecolor='orange',alpha=0.5)
cass.plot(ax=ax,facecolor='none',edgecolor='navy'); plt.show()

# Q3:
centroids_gdf = StateNdProj.copy()
centroids_gdf['geometry'] = StateNdProj.geometry.centroid
fig,ax = plt.subplots(figsize=(10,8))
StateNdProj.plot(ax=ax,facecolor='none',edgecolor='grey',linewidth=0.3)
centroids_gdf.plot(ax=ax,column='COUNTYFP',cmap='tab20',markersize=10)
plt.title('Centroids by County'); plt.show()

# Q4:
simplified = StateNdProj.copy(); simplified['geometry']=StateNdProj.geometry.simplify(1000)
StateNdProj.to_file('output/orig.shp'); simplified.to_file('output/simp.shp')
print('Original:', os.path.getsize('output/orig.shp')//1024,'KB')
print('Simplified:', os.path.getsize('output/simp.shp')//1024,'KB')

# Q5:
total = StateNdProj.groupby('COUNTYFP')['AreaM2'].sum()/1e6
print(total.nlargest(5).round(1))
```
</details>

---

# 6️⃣ Visualizing with Matplotlib

---

> `gdf.plot()` is a GeoPandas wrapper around Matplotlib. Every parameter you know from Matplotlib applies.

| Parameter | Description |
|-----------|-------------|
| `column` | Attribute to colour features by |
| `cmap` | Colormap (e.g. `'RdYlBu'`, `'viridis'`, `'cividis'`) |
| `legend=True` | Show colorbar |
| `facecolor` | Fill colour for polygons |
| `edgecolor` | Border colour |
| `linewidth` | Border thickness |
| `alpha` | Transparency (0–1) |
| `missing_kwds` | Style for features with no data |
| `scheme` | Classification scheme (`'quantiles'`, `'equal_interval'`) |

---

### 💡 Example 6.1 — Basic Map & Choropleth

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt

StateNdProj = gpd.read_file(r'data/tl_2025_38_tract.zip').to_crs(5070)
StateNdProj['AreaKm2'] = StateNdProj.geometry.area / 1e6

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Basic outline map
StateNdProj.plot(ax=ax1, facecolor='none', edgecolor='black', linewidth=0.4)
ax1.set_title('North Dakota Census Tracts (outline)')
ax1.set_axis_off()

# Choropleth by area
StateNdProj.plot(
    ax=ax2, column='AreaKm2',
    cmap='RdYlBu', legend=True,
    legend_kwds={'label': 'Area (km²)', 'shrink': 0.5}
)
ax2.set_title('Tract Area Choropleth (km²)')
ax2.set_axis_off()

plt.tight_layout()
plt.show()

### 💡 Example 6.2 — Styled Map with Custom Colours

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt

StateNd = gpd.read_file(r'data/tl_2025_38_tract.zip').to_crs(5070)
StateNd['AreaKm2'] = StateNd.geometry.area / 1e6

fig, ax = plt.subplots(figsize=(12, 9))

StateNd.plot(
    ax=ax,
    column='AreaKm2',
    cmap='cividis',          # colorblind-friendly
    scheme='quantiles',      # classify into equal quantile bins
    k=5,                     # 5 classes
    legend=True,
    legend_kwds={'title': 'Area (km²)', 'loc': 'lower right'},
    edgecolor='white',
    linewidth=0.3,
    missing_kwds={'color': 'lightgrey', 'label': 'No data'}
)

ax.set_title('North Dakota Census Tract Area\n(5 Quantile Classes)',
             fontsize=14, fontweight='bold')
ax.set_axis_off()
plt.tight_layout()
plt.savefig('output/nd_area_choropleth.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: output/nd_area_choropleth.png')

### ✏️ Exercise 6 — Practice Questions

**Q1.** Create a map of North Dakota tracts coloured by **county** (`COUNTYFP`) using the `'tab20'` categorical colormap. Add a title.

**Q2.** Plot **only Cass County tracts** coloured by `ALAND` with the `'Blues'` colormap. Add a colorbar with the label `'Land Area (m²)'`.

**Q3.** Create a **3-class equal-interval choropleth** of tract area for Grand Forks County. Use a colorblind-safe palette.

**Q4.** Plot all ND tracts in light grey, then **overlay Cass County tracts in red** on the same axes.

**Q5.** Save one of your maps as a **high-resolution PNG** (`dpi=300`) to `output/maps/my_map.png` and confirm the file size.


In [ ]:
# 📝 Write your solutions here

# Q1:

# Q2:

# Q3:

# Q4:

# Q5:


<details>
<summary>🔍 Click to reveal solutions</summary>

```python
import geopandas as gpd, matplotlib.pyplot as plt, os
StateNd = gpd.read_file('data/tl_2025_38_tract.zip').to_crs(5070)
StateNd['AreaKm2'] = StateNd.geometry.area/1e6

# Q1:
fig,ax = plt.subplots(figsize=(12,9))
StateNd.plot(ax=ax,column='COUNTYFP',cmap='tab20',edgecolor='white',linewidth=0.2)
ax.set_title('ND Tracts by County'); ax.set_axis_off(); plt.show()

# Q2:
cass = StateNd[StateNd['COUNTYFP']=='017']
fig,ax = plt.subplots(figsize=(8,7))
cass.plot(ax=ax,column='ALAND',cmap='Blues',legend=True,
          legend_kwds={'label':'Land Area (m²)','shrink':0.6})
ax.set_title('Cass County Tracts — Land Area'); ax.set_axis_off(); plt.show()

# Q3:
gf = StateNd[StateNd['COUNTYFP']=='035']
fig,ax = plt.subplots(figsize=(7,7))
gf.plot(ax=ax,column='AreaKm2',cmap='cividis',scheme='equal_interval',k=3,legend=True)
ax.set_title('Grand Forks County — 3-Class Area'); ax.set_axis_off(); plt.show()

# Q4:
fig,ax = plt.subplots(figsize=(12,9))
StateNd.plot(ax=ax,facecolor='lightgrey',edgecolor='white',linewidth=0.2)
cass.plot(ax=ax,facecolor='red',edgecolor='darkred',linewidth=0.4)
ax.set_title('ND Tracts — Cass County Highlighted'); ax.set_axis_off(); plt.show()

# Q5:
os.makedirs('output/maps',exist_ok=True)
fig.savefig('output/maps/my_map.png',dpi=300,bbox_inches='tight')
print(os.path.getsize('output/maps/my_map.png')//1024,'KB')
```
</details>

---

# 7️⃣ Multi-Panel County Maps (Automation)

---

> Use Python loops + `plt.subplots()` to generate small multiples — one map per county — automatically.

| Concept | Code |
|---------|------|
| Get unique values | `gdf['col'].unique()` |
| Create subplot grid | `fig, axes = plt.subplots(nrows, ncols, figsize=...)` |
| Flatten axes | `axes.flatten()` |
| Hide unused axes | `ax.set_visible(False)` |
| Tight layout | `plt.tight_layout()` |

---

### 💡 Example 7.1 — Plot First 20 Counties in a Grid

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt

StateNdProj = gpd.read_file(r'data/tl_2025_38_tract.zip').to_crs(5070)
StateNdProj['AreaM2'] = StateNdProj.geometry.area

counties = StateNdProj['COUNTYFP'].unique()
print(f'Total unique counties: {len(counties)}')

# 5×4 grid = 20 panels
fig, axes = plt.subplots(5, 4, figsize=(24, 15))
axes_flat = axes.flatten()

for ax, fips in zip(axes_flat, counties[:20]):
    county_gdf = StateNdProj[StateNdProj['COUNTYFP'] == fips]
    county_gdf.plot(
        column='AreaM2', ax=ax,
        cmap='RdYlBu', legend=False
    )
    ax.set_title(f'County {fips}', fontsize=9)
    ax.set_axis_off()

# Hide any unused subplots
for ax in axes_flat[len(counties[:20]):]:
    ax.set_visible(False)

plt.suptitle('North Dakota Counties — Tract Area (first 20)', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('output/county_grid.png', dpi=120, bbox_inches='tight')
plt.show()

### 💡 Example 7.2 — Add County Statistics to Each Panel

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt

StateNdProj = gpd.read_file(r'data/tl_2025_38_tract.zip').to_crs(5070)
StateNdProj['AreaKm2'] = StateNdProj.geometry.area / 1e6

counties = StateNdProj['COUNTYFP'].unique()[:6]   # first 6 for demo

fig, axes = plt.subplots(2, 3, figsize=(15, 10))

for ax, fips in zip(axes.flatten(), counties):
    county_gdf = StateNdProj[StateNdProj['COUNTYFP'] == fips]
    county_gdf.plot(column='AreaKm2', ax=ax, cmap='Blues',
                    edgecolor='black', linewidth=0.5)
    n      = len(county_gdf)
    total  = county_gdf['AreaKm2'].sum()
    ax.set_title(f'County {fips}\n{n} tracts | {total:.0f} km²', fontsize=10)
    ax.set_axis_off()

plt.suptitle('ND County Maps with Statistics', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### ✏️ Exercise 7 — Practice Questions

**Q1.** Plot **all** unique counties (not just 20) in a grid. How many rows and columns do you need? Adjust the figure size accordingly.

**Q2.** For each panel in a 3×3 subplot grid, show a **different attribute** rather than the same one — alternate between `ALAND`, `AWATER`, and `AreaKm2`.

**Q3.** Add a **shared colorbar** for all panels in the 20-county grid (same scale for all). Use `vmin` and `vmax` set to the global min and max of `AreaM2`.

**Q4.** Add a **text annotation** inside each panel showing the county FIPS code and number of tracts, positioned at the centroid of the county.

**Q5.** Save the full county grid as a **PDF** to `output/county_grid.pdf` (use `plt.savefig` with `format='pdf'`). Confirm the file exists.


In [ ]:
# 📝 Write your solutions here

# Q1:

# Q2:

# Q3:

# Q4:

# Q5:


<details>
<summary>🔍 Click to reveal solutions</summary>

```python
import geopandas as gpd, matplotlib.pyplot as plt, numpy as np, math, os
StateNdProj = gpd.read_file('data/tl_2025_38_tract.zip').to_crs(5070)
StateNdProj['AreaKm2'] = StateNdProj.geometry.area/1e6
counties = StateNdProj['COUNTYFP'].unique()

# Q1:
n = len(counties); ncols=5; nrows=math.ceil(n/ncols)
fig,axes = plt.subplots(nrows,ncols,figsize=(ncols*5,nrows*4))
for ax,fips in zip(axes.flatten(),counties):
    StateNdProj[StateNdProj['COUNTYFP']==fips].plot(ax=ax,color='steelblue',edgecolor='k',lw=0.3)
    ax.set_title(f'County {fips}',fontsize=8); ax.set_axis_off()
for ax in axes.flatten()[n:]: ax.set_visible(False)
plt.tight_layout(); plt.show()

# Q3:
vmin = StateNdProj['AreaKm2'].min(); vmax = StateNdProj['AreaKm2'].max()
fig,axes = plt.subplots(5,4,figsize=(24,15))
for ax,fips in zip(axes.flatten(),counties[:20]):
    StateNdProj[StateNdProj['COUNTYFP']==fips].plot(
        column='AreaKm2',ax=ax,cmap='cividis',vmin=vmin,vmax=vmax)
    ax.set_title(f'County {fips}',fontsize=9); ax.set_axis_off()
import matplotlib.cm as cm
sm = cm.ScalarMappable(cmap='cividis',norm=plt.Normalize(vmin=vmin,vmax=vmax))
fig.colorbar(sm,ax=axes,shrink=0.4,label='Area (km²)')
plt.tight_layout(); plt.show()

# Q5:
fig.savefig('output/county_grid.pdf',format='pdf',bbox_inches='tight')
print(os.path.exists('output/county_grid.pdf'))
```
</details>

---

# 8️⃣ Exporting Results

---

> GeoPandas can write to many formats. Choose the right one for your workflow.

| Format | Method | Notes |
|--------|--------|-------|
| Shapefile | `gdf.to_file('out.shp')` | Multiple sidecar files; widely supported |
| GeoJSON | `gdf.to_file('out.geojson', driver='GeoJSON')` | Single file; human-readable |
| GeoPackage | `gdf.to_file('out.gpkg', driver='GPKG')` | Single file; supports layers |
| Parquet | `gdf.to_parquet('out.parquet')` | Fast, columnar; large data |
| CSV | `df.to_csv('out.csv')` | Drop geometry or export as WKT |

---

### 💡 Example 8.1 — Export to Multiple Formats

In [ ]:
import geopandas as gpd
import warnings, os

StateNd     = gpd.read_file(r'data/tl_2025_38_tract.zip')
StateNdProj = StateNd.to_crs(5070)
StateNdProj['AreaKm2']  = StateNdProj.geometry.area / 1e6
StateNdProj['PerimM']   = StateNdProj.geometry.length
StateNdProj['Centroid_WKT'] = StateNdProj.geometry.centroid.to_wkt()

# Drop extra geometry column before saving formats that don't support two
save_gdf = StateNdProj.drop(columns=['Centroid_WKT'] if 'Centroid_WKT' not in StateNdProj.columns else [])

# Shapefile
StateNdProj.to_file('output/nd_tracts_proj.shp')

# GeoPackage
warnings.filterwarnings('ignore')
StateNdProj.to_file('output/nd_tracts_proj.gpkg', driver='GPKG', layer='tracts')

# GeoJSON
StateNdProj.to_file('output/nd_tracts_proj.geojson', driver='GeoJSON')

# Parquet (fastest for large data)
StateNdProj.to_parquet('output/nd_tracts_proj.parquet')

# CSV with WKT geometry
csv_df = StateNdProj.copy()
csv_df['geometry'] = csv_df.geometry.to_wkt()
csv_df.drop(columns=[c for c in csv_df.columns if 'Centroid' in c], errors='ignore').to_csv('output/nd_tracts_proj.csv', index=False)

for fname in ['nd_tracts_proj.shp','nd_tracts_proj.gpkg','nd_tracts_proj.geojson',
              'nd_tracts_proj.parquet','nd_tracts_proj.csv']:
    path = f'output/{fname}'
    if os.path.exists(path):
        print(f'{fname:<35} {os.path.getsize(path)//1024:>6} KB')

### ✏️ Exercise 8 — Practice Questions

**Q1.** Save only **Cass County tracts** to `output/cass_county.geojson`. Re-read it and confirm the feature count.

**Q2.** Save the full ND projected layer to a **GeoPackage** with two layers: one named `'all_tracts'` and another named `'cass_only'`.

**Q3.** Export the projected GeoDataFrame to **Parquet** and **Shapefile**. Compare their file sizes. Which is smaller?

**Q4.** Export a CSV where the geometry is stored as **WKT** in a column called `'wkt_geometry'`. Read it back and reconstruct a GeoDataFrame using `gpd.GeoDataFrame(df, geometry=gpd.GeoSeries.from_wkt(df['wkt_geometry']))`.

**Q5.** Write a helper function `export_gdf(gdf, name, formats=['shp','geojson','gpkg'])` that exports the GeoDataFrame to all specified formats into the `output` folder.


In [ ]:
# 📝 Write your solutions here

# Q1:

# Q2:

# Q3:

# Q4:

# Q5:


<details>
<summary>🔍 Click to reveal solutions</summary>

```python
import geopandas as gpd, os
StateNdProj = gpd.read_file('data/tl_2025_38_tract.zip').to_crs(5070)
cass = StateNdProj[StateNdProj['COUNTYFP']=='017']

# Q1:
cass.to_file('output/cass_county.geojson',driver='GeoJSON')
back = gpd.read_file('output/cass_county.geojson')
print('Feature count match:', len(back)==len(cass))

# Q2:
StateNdProj.to_file('output/nd_layers.gpkg',driver='GPKG',layer='all_tracts')
cass.to_file('output/nd_layers.gpkg',driver='GPKG',layer='cass_only')

# Q3:
StateNdProj.to_parquet('output/nd.parquet')
StateNdProj.to_file('output/nd.shp')
print('Parquet:',os.path.getsize('output/nd.parquet')//1024,'KB')
print('SHP:    ',os.path.getsize('output/nd.shp')//1024,'KB')

# Q4:
import pandas as pd
df = StateNdProj.copy()
df['wkt_geometry'] = df.geometry.to_wkt()
df.drop(columns='geometry').to_csv('output/nd_wkt.csv',index=False)
df2 = pd.read_csv('output/nd_wkt.csv')
gdf_back = gpd.GeoDataFrame(df2,geometry=gpd.GeoSeries.from_wkt(df2['wkt_geometry']),crs=5070)
print(gdf_back.shape, gdf_back.crs)

# Q5:
def export_gdf(gdf, name, formats=None):
    if formats is None: formats=['shp','geojson','gpkg']
    drv = {'shp':'ESRI Shapefile','geojson':'GeoJSON','gpkg':'GPKG'}
    for fmt in formats:
        path = f'output/{name}.{fmt}'
        gdf.to_file(path,driver=drv[fmt])
        print(f'Saved: {path} ({os.path.getsize(path)//1024} KB)')
export_gdf(cass,'cass_export')
```
</details>

---

# 9️⃣ Census Demographic Data with pygris

---

> **pygris** provides a Pythonic interface to the U.S. Census Bureau API.  
> `get_census()` retrieves ACS (American Community Survey) variables for any geography.

| Parameter | Description |
|-----------|-------------|
| `dataset` | Census dataset (e.g. `'acs/acs5/profile'`) |
| `variables` | ACS variable ID (e.g. `'DP02_0067PE'`) |
| `year` | ACS year (e.g. `2021`) |
| `params` | Dict: geography scope (state, county, tract) |
| `return_geoid` | Return GEOID column for joining to spatial data |

> **Useful ACS links:**  
> 🔗 [Census API datasets](https://api.census.gov/data.html)  
> 🔗 [ACS 5-yr Data Profile variables](https://api.census.gov/data/2021/acs/acs5/profile/variables.html)

---

### 💡 Example 9.1 — Install pygris & Retrieve ACS Data

In [ ]:
# !pip install pygris   # uncomment if needed
from pygris.data import get_census

# DP02_0067PE = % Population 25+ with High School Diploma or Higher
nd_education = get_census(
    dataset    = 'acs/acs5/profile',
    variables  = 'DP02_0067PE',
    year       = 2021,
    params     = {'for': 'tract:*', 'in': 'state:38'},
    guess_dtypes  = True,
    return_geoid  = True
)

print(f'Rows returned : {len(nd_education)}')
print(f'Columns       : {nd_education.columns.tolist()}')
nd_education.head()

### 💡 Example 9.2 — Retrieve Multiple Variables

In [ ]:
from pygris.data import get_census

# Retrieve multiple ACS variables at once
nd_multi = get_census(
    dataset   = 'acs/acs5/profile',
    variables = ['DP02_0067PE',   # % HS Graduate+
                 'DP03_0062E',    # Median household income
                 'DP05_0001E'],   # Total population
    year      = 2021,
    params    = {'for': 'tract:*', 'in': 'state:38'},
    guess_dtypes = True,
    return_geoid = True
)

nd_multi.rename(columns={
    'DP02_0067PE': 'pct_hs_grad',
    'DP03_0062E' : 'median_income',
    'DP05_0001E' : 'total_pop'
}, inplace=True)

print(nd_multi[['GEOID','pct_hs_grad','median_income','total_pop']].head())

### ✏️ Exercise 9 — Practice Questions

**Q1.** Use `get_census()` to retrieve `'DP02_0152PE'` (% households with internet access) for all North Dakota tracts. Print the first 5 rows.

**Q2.** Retrieve **median household income** (`'DP03_0062E'`) for ND. What is the statewide **mean**, **median**, and **range**?

**Q3.** Retrieve **total population** (`'DP05_0001E'`) for all ND tracts. Which tract has the **highest population**? Which has the **lowest**?

**Q4.** Retrieve the same education variable (`DP02_0067PE`) for a **different state** (e.g. state code `27` = Minnesota). Compare the mean with North Dakota.

**Q5.** Find the **ACS variable ID** for `'Percent households with a broadband internet subscription'` by browsing the Census API variables page. Retrieve and print the data.


In [ ]:
# 📝 Write your solutions here

# Q1:

# Q2:

# Q3:

# Q4:

# Q5:


<details>
<summary>🔍 Click to reveal solutions</summary>

```python
from pygris.data import get_census

# Q1:
internet = get_census(dataset='acs/acs5/profile',variables='DP02_0152PE',year=2021,
    params={'for':'tract:*','in':'state:38'},guess_dtypes=True,return_geoid=True)
print(internet.head())

# Q2:
income = get_census(dataset='acs/acs5/profile',variables='DP03_0062E',year=2021,
    params={'for':'tract:*','in':'state:38'},guess_dtypes=True,return_geoid=True)
income['DP03_0062E'] = pd.to_numeric(income['DP03_0062E'],errors='coerce')
print('Mean:  ', income['DP03_0062E'].mean())
print('Median:', income['DP03_0062E'].median())
print('Range: ', income['DP03_0062E'].min(),'–',income['DP03_0062E'].max())

# Q3:
pop = get_census(dataset='acs/acs5/profile',variables='DP05_0001E',year=2021,
    params={'for':'tract:*','in':'state:38'},guess_dtypes=True,return_geoid=True)
pop['DP05_0001E'] = pd.to_numeric(pop['DP05_0001E'],errors='coerce')
print('Highest pop tract:', pop.loc[pop['DP05_0001E'].idxmax(),'GEOID'])
print('Lowest pop tract :', pop.loc[pop['DP05_0001E'].idxmin(),'GEOID'])

# Q4:
mn_edu = get_census(dataset='acs/acs5/profile',variables='DP02_0067PE',year=2021,
    params={'for':'tract:*','in':'state:27'},guess_dtypes=True,return_geoid=True)
nd_edu = get_census(dataset='acs/acs5/profile',variables='DP02_0067PE',year=2021,
    params={'for':'tract:*','in':'state:38'},guess_dtypes=True,return_geoid=True)
print('MN mean HS grad%:', pd.to_numeric(mn_edu['DP02_0067PE'],errors='coerce').mean())
print('ND mean HS grad%:', pd.to_numeric(nd_edu['DP02_0067PE'],errors='coerce').mean())
```
</details>

---

# 🔟 Choropleth Mapping with Census Data

---

> Join ACS tabular data to the spatial GeoDataFrame using **GEOID** as the common key,  
> then visualize with a polished choropleth map.

| Step | Code |
|------|------|
| Merge | `gdf.merge(df, on='GEOID', how='inner')` |
| Basic plot | `merged.plot(column='var', legend=True)` |
| Styled plot | custom `figsize`, `cmap`, `edgecolor`, `legend_kwds` |
| Colorblind palette | `'cividis'`, `'viridis'`, `'RdYlBu'` |

---

### 💡 Example 10.1 — Merge ACS Data & Basic Choropleth

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
from pygris.data import get_census

# Load spatial data
StateNdProj = gpd.read_file(r'data/tl_2025_38_tract.zip').to_crs(5070)

# Load ACS education variable
nd_edu = get_census(
    dataset='acs/acs5/profile', variables='DP02_0067PE', year=2021,
    params={'for': 'tract:*', 'in': 'state:38'},
    guess_dtypes=True, return_geoid=True
)

# Merge on GEOID
ND_merged = StateNdProj.merge(nd_edu, how='inner', on='GEOID')
print(f'Merged rows: {len(ND_merged)}')

# Quick choropleth
ND_merged.plot(
    column='DP02_0067PE', legend=True,
    legend_kwds={'shrink': 0.4},
    figsize=(10, 8)
)
plt.title('% High School Graduate (Population 25+) — North Dakota')
plt.show()

### 💡 Example 10.2 — Polished Publication-Ready Map

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
from pygris.data import get_census

StateNdProj = gpd.read_file(r'data/tl_2025_38_tract.zip').to_crs(5070)
nd_edu      = get_census(dataset='acs/acs5/profile', variables='DP02_0067PE', year=2021,
                         params={'for':'tract:*','in':'state:38'},
                         guess_dtypes=True, return_geoid=True)
ND_merged   = StateNdProj.merge(nd_edu, how='inner', on='GEOID')

fig, ax = plt.subplots(figsize=(14, 10))

ND_merged.plot(
    ax=ax,
    column='DP02_0067PE',
    cmap='cividis',             # colorblind-friendly
    linewidth=0.5,
    edgecolor='white',
    legend=True,
    legend_kwds={'shrink': 0.5, 'label': '% HS Graduate (Pop 25+)'},
    missing_kwds={'color': 'lightgrey', 'label': 'No data'}
)

ax.set_title(
    '% Population 25+ with High School Diploma or Higher\nNorth Dakota — ACS 2021 (5-Year)',
    fontsize=15, fontweight='bold', pad=15
)
ax.annotate('Source: U.S. Census Bureau, ACS 2021',
            xy=(0.01, 0.01), xycoords='axes fraction', fontsize=9, color='grey')
ax.axis('off')
plt.tight_layout()
plt.savefig('output/nd_education_map.png', dpi=200, bbox_inches='tight')
plt.show()
print('Saved: output/nd_education_map.png')

### 💡 Example 10.3 — Internet Access Map (Exercise Variable)

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
from pygris.data import get_census

StateNdProj = gpd.read_file(r'data/tl_2025_38_tract.zip').to_crs(5070)

# DP02_0152PE = % Households with computer and internet access
nd_internet = get_census(
    dataset='acs/acs5/profile', variables='DP02_0152PE', year=2021,
    params={'for': 'tract:*', 'in': 'state:38'},
    guess_dtypes=True, return_geoid=True
)

ND_merged2 = StateNdProj.merge(nd_internet, how='inner', on='GEOID')

fig, ax = plt.subplots(figsize=(14, 10))
ND_merged2.plot(
    ax=ax, column='DP02_0152PE',
    cmap='YlGnBu', linewidth=0.4, edgecolor='white',
    legend=True,
    legend_kwds={'shrink': 0.5, 'label': '% Households — Computers & Internet'},
    missing_kwds={'color': 'lightgrey'}
)
ax.set_title('% Households with Computer & Internet Access\nNorth Dakota — ACS 2021 (5-Year)',
             fontsize=15, fontweight='bold')
ax.annotate('Source: U.S. Census Bureau, ACS 2021',
            xy=(0.01, 0.01), xycoords='axes fraction', fontsize=9, color='grey')
ax.axis('off')
plt.tight_layout()
plt.savefig('output/nd_internet_map.png', dpi=200, bbox_inches='tight')
plt.show()

### ✏️ Exercise 10 — Practice Questions

**Q1.** Merge ACS **median household income** (`DP03_0062E`) to the ND GeoDataFrame and create a polished choropleth. Add a title, colorbar label, and source annotation.

**Q2.** After merging, filter the merged GeoDataFrame to **Cass County only** and create an internet-access choropleth for just that county.

**Q3.** Create a **side-by-side comparison** (`plt.subplots(1, 2)`) of the education rate and internet access maps at the same spatial extent.

**Q4.** Add **county boundary outlines** (dissolve by `COUNTYFP` first) on top of the tract-level choropleth for context.

**Q5.** Retrieve and map **block group level** data for Cass County: use `'for': 'block group:*', 'in': 'state:38 county:017'` and plot internet access.


In [ ]:
# 📝 Write your solutions here

# Q1:

# Q2:

# Q3:

# Q4:

# Q5:


<details>
<summary>🔍 Click to reveal solutions</summary>

```python
import geopandas as gpd, matplotlib.pyplot as plt, pandas as pd
from pygris.data import get_census
StateNdProj = gpd.read_file('data/tl_2025_38_tract.zip').to_crs(5070)

# Q1:
inc = get_census(dataset='acs/acs5/profile',variables='DP03_0062E',year=2021,
    params={'for':'tract:*','in':'state:38'},guess_dtypes=True,return_geoid=True)
merged_inc = StateNdProj.merge(inc,on='GEOID',how='inner')
merged_inc['DP03_0062E'] = pd.to_numeric(merged_inc['DP03_0062E'],errors='coerce')
fig,ax = plt.subplots(figsize=(14,10))
merged_inc.plot(ax=ax,column='DP03_0062E',cmap='RdYlGn',legend=True,
    legend_kwds={'shrink':0.5,'label':'Median HH Income ($)'},
    edgecolor='white',linewidth=0.3,missing_kwds={'color':'lightgrey'})
ax.set_title('Median Household Income — ND ACS 2021',fontsize=14,fontweight='bold')
ax.annotate('Source: U.S. Census Bureau',xy=(0.01,0.01),xycoords='axes fraction',fontsize=9,color='grey')
ax.axis('off'); plt.tight_layout(); plt.show()

# Q4 — county boundaries overlay:
county_bounds = StateNdProj.dissolve(by='COUNTYFP').reset_index()
fig,ax = plt.subplots(figsize=(14,10))
merged_inc.plot(ax=ax,column='DP03_0062E',cmap='RdYlGn',legend=True,
    edgecolor='none',missing_kwds={'color':'lightgrey'})
county_bounds.plot(ax=ax,facecolor='none',edgecolor='black',linewidth=0.8)
ax.set_title('Median Income with County Boundaries'); ax.axis('off'); plt.show()
```
</details>

---

# 1️⃣1️⃣ Bonus — Spatial Joins & Dissolve

---

> **Spatial join** assigns attributes from one layer to another based on spatial relationships.  
> **Dissolve** merges geometries sharing a common attribute into one polygon.

| Operation | Code | Predicate options |
|-----------|------|-------------------|
| Spatial join | `gpd.sjoin(left, right, how='left', predicate='within')` | `within`, `intersects`, `contains` |
| Dissolve | `gdf.dissolve(by='column', aggfunc='sum')` | `sum`, `mean`, `count`, `first` |
| Union all | `gdf.dissolve()` | (no by= → one polygon) |

---

### 💡 Example 11.1 — Dissolve Tracts into Counties

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt

StateNdProj = gpd.read_file(r'data/tl_2025_38_tract.zip').to_crs(5070)
StateNdProj['AreaKm2'] = StateNdProj.geometry.area / 1e6

# Dissolve all tracts into counties — summing ALAND and AreaKm2
counties = StateNdProj.dissolve(
    by='COUNTYFP',
    aggfunc={'ALAND': 'sum', 'AreaKm2': 'sum', 'TRACTCE': 'count'}
).rename(columns={'TRACTCE': 'num_tracts'}).reset_index()

print(f'Counties dissolved: {len(counties)}')
print(counties[['COUNTYFP','num_tracts','AreaKm2']].head())

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
StateNdProj.plot(ax=ax1, facecolor='none', edgecolor='grey', linewidth=0.3)
ax1.set_title('Census Tracts'); ax1.set_axis_off()

counties.plot(ax=ax2, column='num_tracts', cmap='YlOrRd', legend=True,
              edgecolor='black', linewidth=0.5)
ax2.set_title('Counties — Tract Count'); ax2.set_axis_off()

plt.tight_layout()
plt.show()

### 💡 Example 11.2 — Spatial Join (Points in Polygons)

In [ ]:
import geopandas as gpd
from shapely.geometry import Point
import numpy as np
import matplotlib.pyplot as plt

StateNdProj = gpd.read_file(r'data/tl_2025_38_tract.zip').to_crs(5070)

# Create 50 random points within ND bounding box
rng = np.random.default_rng(42)
minx, miny, maxx, maxy = StateNdProj.total_bounds
rand_pts = gpd.GeoDataFrame(
    {'id': range(50)},
    geometry=[Point(rng.uniform(minx, maxx), rng.uniform(miny, maxy)) for _ in range(50)],
    crs=StateNdProj.crs
)

# Spatial join: find which tract each point falls in
joined = gpd.sjoin(rand_pts, StateNdProj[['GEOID','COUNTYFP','geometry']],
                   how='left', predicate='within')

print(f'Points with a matching tract: {joined["GEOID"].notna().sum()}')
print(joined[['id','GEOID','COUNTYFP']].head())

fig, ax = plt.subplots(figsize=(10, 8))
StateNdProj.plot(ax=ax, facecolor='lightyellow', edgecolor='grey', linewidth=0.3)
rand_pts.plot(ax=ax, color='red', markersize=20, zorder=5)
ax.set_title('Random Points Spatially Joined to ND Tracts')
ax.set_axis_off()
plt.show()

### ✏️ Exercise 11 — Practice Questions

**Q1.** Dissolve the ND GeoDataFrame by county (`COUNTYFP`). Plot the resulting county-level map, coloured by total `ALAND` (km²).

**Q2.** Create a GeoDataFrame of 5 ND city points (Fargo, Bismarck, Grand Forks, Minot, Williston — look up coordinates). Spatial-join them to the tract layer to find which `GEOID` each city falls in.

**Q3.** Use `gpd.sjoin()` with `predicate='intersects'` to find all tracts that **intersect** a 20 km buffer around Fargo. How many tracts are in the buffer zone?

**Q4.** Dissolve all ND tracts into a **single state outline** polygon. Plot it on top of a satellite basemap using `contextily` (install if needed).

**Q5.** For each county, use dissolve to compute the **mean** and **total** land area, then create a bar chart of the top 10 counties by total area.


In [ ]:
# 📝 Write your solutions here

# Q1:

# Q2:

# Q3:

# Q4:

# Q5:


<details>
<summary>🔍 Click to reveal solutions</summary>

```python
import geopandas as gpd, numpy as np, matplotlib.pyplot as plt
from shapely.geometry import Point
StateNdProj = gpd.read_file('data/tl_2025_38_tract.zip').to_crs(5070)
StateNdProj['AreaKm2'] = StateNdProj.geometry.area/1e6

# Q1:
counties = StateNdProj.dissolve(by='COUNTYFP',aggfunc={'AreaKm2':'sum'}).reset_index()
fig,ax = plt.subplots(figsize=(10,8))
counties.plot(ax=ax,column='AreaKm2',cmap='YlOrRd',legend=True,edgecolor='black',lw=0.5)
ax.set_title('ND Counties by Total Land Area (km²)'); ax.axis('off'); plt.show()

# Q2:
cities = gpd.GeoDataFrame({'city':['Fargo','Bismarck','Grand Forks','Minot','Williston'],
    'geometry':[Point(-96.79,46.88),Point(-100.78,46.81),Point(-97.03,47.93),
               Point(-101.30,48.23),Point(-103.62,48.15)]},crs='EPSG:4326').to_crs(StateNdProj.crs)
joined = gpd.sjoin(cities,StateNdProj[['GEOID','geometry']],how='left',predicate='within')
print(joined[['city','GEOID']])

# Q3:
fargo = gpd.GeoDataFrame(geometry=[Point(-96.79,46.88)],crs='EPSG:4326').to_crs(StateNdProj.crs)
fargo_buf = fargo.copy(); fargo_buf['geometry'] = fargo.geometry.buffer(20000)
intersects = gpd.sjoin(StateNdProj,fargo_buf,how='inner',predicate='intersects')
print('Tracts in 20km Fargo buffer:', len(intersects))

# Q5:
top10 = counties.nlargest(10,'AreaKm2')[['COUNTYFP','AreaKm2']]
fig,ax = plt.subplots(figsize=(10,5))
ax.bar(top10['COUNTYFP'],top10['AreaKm2'],color='steelblue')
ax.set_xlabel('County FIPS'); ax.set_ylabel('Area (km²)')
ax.set_title('Top 10 ND Counties by Land Area'); plt.show()
```
</details>

---

# 📝 Chapter Review Questions

---

Answer the following questions in the cell below.

**1.** What is the fundamental difference between a Pandas `DataFrame` and a GeoPandas `GeoDataFrame`?  
**2.** Why is it necessary to reproject from EPSG:4269 to EPSG:5070 before calculating area?  
**3.** What does `gdf.dissolve(by='COUNTYFP', aggfunc='sum')` do? What is the output shape?  
**4.** What is the `GEOID` column and why is it used as the join key between ACS data and TIGER shapefiles?  
**5.** What is the difference between `gpd.sjoin()` with `predicate='within'` vs `predicate='intersects'`?  
**6.** What does the `missing_kwds` parameter in `gdf.plot()` control?  
**7.** Why is `'cividis'` recommended over `'jet'` for choropleth maps?  
**8.** What does `gdf.geometry.buffer(0)` do, and when would you use it?

---

In [ ]:
# 📝 Write your answers here as comments

# 1:

# 2:

# 3:

# 4:

# 5:

# 6:

# 7:

# 8:


---

# 🎉 Congratulations — All 12 Sections Complete!

---

| ✅ | Section |
|----|----------|
| ✅ | 0 — Setup & Environment |
| ✅ | 1 — Reading Vector Data |
| ✅ | 2 — Exploring & Inspecting a GeoDataFrame |
| ✅ | 3 — Filtering & Subsetting Spatial Data |
| ✅ | 4 — Reprojecting for Accurate Analysis |
| ✅ | 5 — Spatial Operations & Attribute Manipulation |
| ✅ | 6 — Visualizing with Matplotlib |
| ✅ | 7 — Multi-Panel County Maps |
| ✅ | 8 — Exporting Results |
| ✅ | 9 — Census Demographic Data with pygris |
| ✅ | 10 — Choropleth Mapping with Census Data |
| ✅ | 11 — Bonus: Spatial Joins & Dissolve |

---

## 🚀 Next Steps

| Topic | Libraries |
|-------|-----------|
| Network analysis | `osmnx`, `networkx` |
| Spatial statistics | `pysal`, `esda`, `libpysal` |
| Interactive maps | `folium`, `leafmap`, `kepler.gl` |
| 3D vector viz | `pydeck`, `plotly` |
| Raster + Vector combined | `rasterio`, `rasterstats` |

---

### Happy Mapping! 🗺️💻

*Every polygon has a story — GeoPandas helps you tell it.*

---